# TIDES — EDS paper reproduction (Tables 1 & 4)

This notebook runs the **existing repository scripts**, without changing the TIDES implementation. It starts with one EDS sequence and can be repeated across sequences. **Paper numbers are EDS dataset averages**, so a single sequence is only a smoke test, not a validated reproduction. The repo uses some assumed metric/hyperparameter settings; matching numbers is not guaranteed.

Run Jupyter from the repository root, on a machine with 4DGaussians dependencies and GPU access.

In [ ]:
from pathlib import Path
import json, subprocess, sys, shlex
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path.cwd().resolve()
assert (ROOT/'scripts'/'simulate.py').exists(), 'Launch Jupyter from repository root'
def run(*cmd):
    cmd=[str(v) for v in cmd]
    print('$',shlex.join(cmd),flush=True)
    subprocess.run(cmd,cwd=ROOT,check=True)
print('Repository:',ROOT)

## 1. Settings

Change `EDS_ROOT` to your data directory. Use the existing prepared scene/checkpoint whenever possible. Every costly stage is **off by default**. Set a flag to `True` only for the stage you intend to run.

In [ ]:
EDS_ROOT=Path('/u/marmary/airlab_v2e/evtween/data/eds')
SEQUENCE='00_peanuts_dark'
WORK=ROOT/'work'/'eds'/SEQUENCE
FOURDGS=ROOT/'third_party'/'4DGaussians'
T_START,T_END=0.0,1.5
SEED=0
DO_INSPECT=True
DO_PREPARE=False
DO_TRAIN=False
DO_TIDES=False
DO_BASELINES=False
DO_EVALUATE=False
DO_ABLATIONS=False
METHODS=['esim','v2e','iebcs','dvs_voltmeter','tides_10x','tides_adaptive']
print('EDS:',EDS_ROOT.exists(),'prepared:',(WORK/'tides_meta.json').exists(),'checkpoint:',(WORK/'4dgs'/'point_cloud').exists())

## 2. Inspect EDS and prepare scene

The repository's EDS loader expects its own dataset structure and calibration. If inspection fails, resolve that before training. Preparation writes to `WORK`.

In [ ]:
if DO_INSPECT:
    assert EDS_ROOT.exists(),f'EDS directory missing: {EDS_ROOT}'
    run(sys.executable,'scripts/inspect_dataset.py','--dataset','eds','--root',EDS_ROOT,'--seq',SEQUENCE)
if DO_PREPARE:
    run(sys.executable,'scripts/prepare_scene.py','--dataset','eds','--root',EDS_ROOT,'--seq',SEQUENCE,'--out',WORK,'--t-start',T_START,'--t-end',T_END)

## 3. Train 4DGS (only if no checkpoint)

This can be expensive. Uses the same command as `scripts/run_sequence.sh`. The notebook does not retrain automatically.

In [ ]:
if DO_TRAIN:
    assert (WORK/'scene').exists(), 'Prepare scene first'
    assert (FOURDGS/'train.py').exists(), '4DGaussians checkout not found'
    run(sys.executable,FOURDGS/'train.py','-s',WORK/'scene','-m',WORK/'4dgs','--configs',ROOT/'configs'/'4dgs'/'tides_default.py','--port',6017)

## 4. Simulate TIDES (10× and adaptive)

Reuses one 4DGS checkpoint. Outputs `events/tides_10x.h5` and `events/tides_adaptive.h5`.

In [ ]:
if DO_TIDES:
    assert (WORK/'4dgs'/'point_cloud').exists(), 'Missing trained 4DGS checkpoint'
    for name in ['tides_10x','tides_adaptive']:
        run(sys.executable,'scripts/simulate.py','--work',WORK,'--fourdgs',FOURDGS,'--config',ROOT/'configs'/(name+'.yaml'),'--name',name,'--seed',SEED)

## 5. Optional baselines

Runs ESIM, V2E, IEBCS, and DVS-Voltmeter on the same rendered frames. These require their upstream dependencies.

In [ ]:
if DO_BASELINES:
    run(sys.executable,'scripts/run_baselines.py','--work',WORK,'--fourdgs',FOURDGS,'--third-party',ROOT/'third_party','--methods','esim','v2e','iebcs','dvs_voltmeter')

## 6. Evaluate Table 1 metrics

IG-NLL and Chamfer are computed against EDS real events. **Metric definitions in `configs/metrics.yaml` are documented assumptions**, so discrepancies from the paper may be methodological.

In [ ]:
if DO_EVALUATE:
    available=[x for x in METHODS if (WORK/'events'/(x+'.h5')).exists()]
    assert available,'No simulated event files available'
    run(sys.executable,'scripts/evaluate.py','--work',WORK,'--fourdgs',FOURDGS,'--methods',*available)
metrics_path=WORK/'metrics.json'
results=json.loads(metrics_path.read_text()) if metrics_path.exists() else {}
rows=[{'method':name,**results.get('methods',{}).get(name,{})} for name in METHODS]
df=pd.DataFrame(rows).set_index('method')
display(df)
print('Missing event/metric results:',[x for x in METHODS if x not in results.get('methods',{})])

## 7. Optional Table 4 ablations

These rerun simulation with the repository's ablation overrides; expensive and **not required for the first smoke test**.

In [ ]:
if DO_ABLATIONS:
    assert (WORK/'4dgs'/'point_cloud').exists()
    run(sys.executable,'scripts/run_ablations.py','--work',WORK,'--fourdgs',FOURDGS)

## 8. Compare with paper-reported EDS averages

**Do not interpret a single-sequence discrepancy as a failed reproduction.** This compares one run with dataset means solely for orientation. The `paper_numbers.py` values are repository-transcribed numbers, not independently verified here.

In [ ]:
from tides.paper_numbers import TABLE1, TABLE4
comparison=[]
for name,vals in results.get('methods',{}).items():
    if name in TABLE1 and 'eds' in TABLE1[name]:
        p_ig,p_ch=TABLE1[name]['eds']
        comparison.append({'method':name,'ours IG-NLL':vals.get('ig_nll'),'paper EDS IG-NLL':p_ig,'ours Chamfer':vals.get('chamfer'),'paper EDS Chamfer':p_ch})
comparison=pd.DataFrame(comparison)
if len(comparison):
    display(comparison)
    comparison.set_index('method')[['ours Chamfer','paper EDS Chamfer']].plot.bar(figsize=(10,4),title='One-sequence vs dataset average (not directly comparable)')
    plt.tight_layout();plt.show()
else: print('No results yet. Run the selected stages above.')
print('Table 4 paper reference:')
display(pd.DataFrame([{'variant':k,'label':v[0],'paper IG-NLL':v[1],'paper Chamfer':v[2]} for k,v in TABLE4.items()]))

## 9. Aggregate multiple EDS sequences and save reproducibility record

For paper-level comparison, prepare/train/simulate/evaluate **every corresponding EDS sequence** in separate `work/eds/<sequence>` folders, then aggregate using `make_tables.py`. Document the exact sequence list, seeds, checkpoint details, and metric assumptions.

In [ ]:
WORK_ROOT=ROOT/'work'
if list(WORK_ROOT.glob('eds/*/metrics.json')):
    run(sys.executable,'scripts/make_tables.py','--work-root',WORK_ROOT,'--out',ROOT/'results'/'tables.md')
    print('Saved results/tables.md')
else: print('No EDS metrics found yet; aggregation skipped.')
manifest={'sequence':SEQUENCE,'dataset_root':str(EDS_ROOT),'work':str(WORK),'seed':SEED,'requested_methods':METHODS,'available_methods':list(results.get('methods',{})),'metric_config':results.get('metric_config'),'caveat':'Independent implementation; paper dataset means require matching sequence coverage and evaluation definitions.'}
print(json.dumps(manifest,indent=2,default=str))
if metrics_path.exists():
    path=WORK/'reproduction_manifest.json';path.write_text(json.dumps(manifest,indent=2,default=str));print('Saved:',path)